# Markov Chain Text Generator — PRODIGY_GA_Task3

A statistical text-generation system based on **Markov Chains**. The model learns which words or characters are likely to follow a given sequence of previous tokens and uses those transition patterns to generate new text.

**Task:** Implement a text generation model using Markov Chains.

**Key features:** word-level generation, character-level generation, configurable Markov order, reproducible results, and an order comparison experiment.

No GPU is required — the implementation uses Python and runs on a CPU.

## Step 1: Define the Markov chain class

In [ ]:
import random
from collections import defaultdict

random.seed(42)
class MarkovChain:
    """
    A generic Markov chain model for text generation.

    Parameters
    ----------
    order : int
        The number of previous tokens (the "state") used to predict the
        next token. Higher order = more coherent but less varied output,
        and needs more training text to avoid gaps in the model.
    level : str
        Either "word" or "char" — determines whether tokens are words
        or individual characters.
    """

    def __init__(self, order=2, level="word"):
        if level not in ("word", "char"):
            raise ValueError("level must be 'word' or 'char'")
        self.order = order
        self.level = level
        # Maps a state (tuple of previous tokens) -> list of possible next tokens
        self.model = defaultdict(list)
        self.start_states = []

    def _tokenize(self, text):
        if self.level == "word":
            return text.split()
        return list(text)

    def train(self, text):
        """Build the transition table (state -> list of next tokens) from training text."""
        tokens = self._tokenize(text)

        if len(tokens) <= self.order:
            raise ValueError("Training text is too short for the given order.")

        for i in range(len(tokens) - self.order):
            state = tuple(tokens[i:i + self.order])
            next_token = tokens[i + self.order]
            self.model[state].append(next_token)

        self.start_states = list(self.model.keys())

    def generate(self, length=50, seed=None):
        """
        Generate new text.

        Parameters
        ----------
        length : int
            Number of tokens (words or characters) to generate.
        seed : tuple or None
            Optional starting state (tuple of `order` tokens). If None,
            a random known state is chosen.
        """
        if not self.model:
            raise RuntimeError("Model has not been trained yet. Call train() first.")

        state = seed if seed is not None else random.choice(self.start_states)
        result = list(state)

        for _ in range(length - self.order):
            choices = self.model.get(state)
            if not choices:
                state = random.choice(self.start_states)
                choices = self.model[state]

            next_token = random.choice(choices)
            result.append(next_token)
            state = tuple(result[-self.order:])

        separator = " " if self.level == "word" else ""
        return separator.join(result)

## Step 2: Provide training text

Use the sample text below, or replace it with your own (paste text directly, or upload a `.txt` file — see the optional cell further down).

In [ ]:
sample_text = """
Artificial intelligence is transforming the way people interact with technology. Machine learning systems learn patterns from data and use those patterns to make predictions. Generative AI can create text, images, music, and other forms of content. Natural language processing helps computers understand and generate human language. Text generation is one application of language modeling where a system predicts what may come next. Markov chains provide a simple statistical approach to this problem by using previous tokens as context. A higher order Markov chain can capture more context, while a lower order model can produce more varied combinations. Good training text is important because the model can only generate transitions that it has learned from the input corpus. Experimenting with different orders helps us understand the relationship between context, coherence, and variation.
"""
print("Training corpus characters:", len(sample_text))

### About the training corpus

The notebook uses a small built-in corpus so it can run immediately. You can replace `sample_text` with a larger `.txt` corpus for better and more diverse generation.

### Optional: upload your own `.txt` file instead

Run this cell, click "Choose Files", and select a text file. It will overwrite `sample_text` above with your file's content.

## Step 3: Train and generate — word-level

In [ ]:
word_model = MarkovChain(order=2, level="word")
word_model.train(sample_text)
print(word_model.generate(length=30))

### Word-level examples

Using a fixed random seed makes the demonstration reproducible.

In [ ]:
for seed in [("artificial", "intelligence"), ("machine", "learning"), ("text", "generation")]:
    if seed in word_model.model:
        print(f"Seed: {' '.join(seed)}")
        print(word_model.generate(length=25, seed=seed))
        print()
    else:
        print(f"Seed not found: {' '.join(seed)}")

## Step 4: Train and generate — character-level

In [ ]:
char_model = MarkovChain(order=4, level="char")
char_model.train(sample_text)
print(char_model.generate(length=200))

## Step 5: Compare different Markov orders

The order controls how many previous tokens are used as the state. Lower orders generally provide more flexibility, while higher orders preserve more local context when the training corpus contains enough examples.

In [ ]:
print("WORD-LEVEL ORDER COMPARISON\n")
for order in [1, 2, 3]:
    model = MarkovChain(order=order, level="word")
    model.train(sample_text)
    print(f"Order {order}: {model.generate(length=25)}\n")

## Step 6: Key observations

- **Order 1:** uses one previous token, so output can be more varied but may lose context.
- **Order 2:** uses two previous tokens and usually preserves more local structure.
- **Order 3:** uses three previous tokens and can be more context-aware, but requires enough training data to avoid unseen states.
- The generator is **not a neural network or transformer**; it learns a transition table directly from the corpus.
- Larger and more diverse training text generally provides more useful transitions.

## Conclusion

This project demonstrates how a Markov Chain can be used for basic text generation at both word and character levels. It provides a lightweight way to understand the foundations of probabilistic sequence generation before moving to more advanced neural language models.

## Notes

- **`order`**: higher = more coherent but more repetitive (fewer novel combinations); lower = more random but more varied.
- **More training text = better results.** The tiny sample above is just for demonstration — try uploading a book, article, or large corpus for more interesting output.
- **No GPU needed.** This algorithm is pure Python/CPU and runs instantly even on Colab's default runtime.